In [2]:
import sys
from pathlib import Path

# Add the project root to Python's path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(project_root)

c:\kaim\brent-oil-change-point-analysis


In [3]:
import importlib
import src.data_loader

importlib.reload(src.data_loader)

from src.data_loader import load_data


In [4]:
import inspect

print(inspect.getsource(load_data))

def load_data(filepath):
    """
    Load Brent oil price dataset.
    """

    try:
        df = pd.read_csv(filepath)

        df["Date"] = pd.to_datetime(
            df["Date"],
            format="mixed"
        )

        df = df.sort_values("Date")

        return df

    except Exception as e:
        raise Exception(f"Error loading dataset: {e}")



In [5]:
import pandas as pd
print(pd.__version__)

2.3.3


In [6]:
df = load_data("../data/brent_oil_prices.csv")

print(df.head())
print(df.tail())

        Date  Price
0 1987-05-20  18.63
1 1987-05-21  18.45
2 1987-05-22  18.55
3 1987-05-25  18.60
4 1987-05-26  18.63
           Date  Price
9006 2022-11-08  96.85
9007 2022-11-09  93.05
9008 2022-11-10  94.25
9009 2022-11-11  96.37
9010 2022-11-14  93.59


In [7]:
weekly = (
    df.set_index("Date")
      .resample("W")["Price"]
      .mean()
      .dropna()
      .reset_index()
)

print(weekly.head())
print(len(weekly))

        Date      Price
0 1987-05-24  18.543333
1 1987-05-31  18.602000
2 1987-06-07  18.702000
3 1987-06-14  18.754000
4 1987-06-21  19.007500
1853


In [9]:
prices = weekly["Price"].values

In [13]:
from src.change_point import (
    build_change_point_model,
    run_sampling,
    summarize_results
)

In [14]:
model = build_change_point_model(prices)

trace = run_sampling(model)

### Note

On some Windows environments PyTensor may fail to compile C extensions due to compiler incompatibilities
(e.g., legacy MinGW installations). The model definition is complete and follows the project
requirements. Sampling was successfully verified on a minimal PyMC example in the same environment,
indicating the remaining issue is environment-specific rather than a modeling issue.

In [ ]:
az.summary(trace)

az.plot_trace(trace)

az.plot_posterior(trace, var_names=["tau","mu_1","mu_2","sigma"])